In [ ]:
#@title 🎧 Download Narration Audio & Play Introduction
import os as _os
if not _os.path.exists("/content/narration"):
    !pip install -q gdown
    import gdown
    gdown.download(id="1oYuEzYAK4CxDP2GsyS_brsW4LFBEgnV8", output="/content/narration.zip", quiet=False)
    !unzip -q /content/narration.zip -d /content/narration
    !rm /content/narration.zip
    print(f"Loaded {len(_os.listdir('/content/narration'))} narration segments")
else:
    print("Narration audio already loaded.")

from IPython.display import Audio, display
display(Audio("/content/narration/05_00_intro.mp3"))

In [ ]:
#@title 🎧 Listen: Setup Gpu
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/05_01_setup_gpu.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

In [ ]:
#@title 🎧 Listen: Closing
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/05_38_closing.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

In [ ]:
# 🔧 Setup: Run this cell first!
# Check GPU availability and install dependencies

import torch
import sys

# Check GPU
if torch.cuda.is_available():
    device = torch.device('cuda')
    print(f"✅ GPU available: {torch.cuda.get_device_name(0)}")
    print(f"   Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    device = torch.device('cpu')
    print("⚠️ No GPU detected. Some cells may run slowly.")
    print("   Go to Runtime → Change runtime type → GPU")

print(f"\n📦 Python {sys.version.split()[0]}")
print(f"🔥 PyTorch {torch.__version__}")

# Set random seeds for reproducibility
import random
import numpy as np

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print(f"🎲 Random seed set to {SEED}")

%matplotlib inline

In [ ]:
#@title 🎧 Listen: Notebook Overview
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/05_02_notebook_overview.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

# RAG Evaluation and the Complete Pipeline

*Part 5 (Capstone) of the Vizuara series on RAG Systems*
*Estimated time: 60 minutes*

You have spent four notebooks building every component of a RAG system from first principles — chunking strategies that preserve meaning, embedding models that capture semantics, vector databases that enable sub-millisecond search, and retrieval strategies that balance precision with recall. Now comes the question that separates prototypes from production systems: **how do you know if it actually works?**

A RAG pipeline that "seems to work" on a handful of test queries is not ready for deployment. It might retrieve the right documents but hallucinate details that are not in them. It might generate correct answers while wasting context tokens on irrelevant chunks. It might appear faithful while silently blending parametric knowledge with retrieved context — the worst of both worlds.

In this capstone, we will build a rigorous evaluation framework around the RAGAS metrics (faithfulness, context relevance, and answer correctness), implement each scorer from scratch, run a full pipeline ablation study, and construct a decision framework for choosing between RAG, fine-tuning, and long context. By the end, you will have a complete, evaluated RAG system — and the tools to diagnose exactly where it fails.

In [ ]:
#@title 🎧 Listen: Ai Assistant
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/05_03_ai_assistant.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

# 🤖 AI Teaching Assistant

Need help with this notebook? Open the **AI Teaching Assistant** — it has already read this entire notebook and can help with concepts, code, and exercises.

**[👉 Open AI Teaching Assistant](https://pods.vizuara.ai/courses/rag-systems/practice/5/assistant)**

*Tip: Open it in a separate tab and work through this notebook side-by-side.*


In [ ]:
#@title 🎧 Listen: Install Deps
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/05_04_install_deps.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

In [ ]:
# Section 1 — Setup: install all dependencies

!pip install -q sentence-transformers faiss-cpu anthropic rank_bm25 numpy matplotlib pandas

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import pandas as pd
import re
import json
import time
import textwrap
from typing import List, Dict, Tuple, Optional, Any
from collections import Counter
from dataclasses import dataclass, field

%matplotlib inline
plt.rcParams['figure.dpi'] = 120
plt.rcParams['font.size'] = 11
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False

np.random.seed(42)

print("Setup complete! All dependencies installed.")

In [ ]:
#@title 🎧 Listen: Why Matter Concept
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/05_05_why_matter_concept.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

## 1. Why Does This Matter?

A RAG system that cannot be measured cannot be improved. Let us make this concrete with a failure case that looks correct on the surface but is deeply problematic.

Consider a question about climate policy: *"What percentage of global CO2 emissions come from transportation?"*

Your RAG pipeline retrieves a chunk that says: *"Transportation accounts for approximately 16% of global greenhouse gas emissions, with road vehicles responsible for nearly three-quarters of transport emissions."*

The model generates: *"Transportation accounts for approximately 16% of global CO2 emissions. Road vehicles are responsible for about 75% of this, while aviation contributes roughly 12% and shipping adds about 11%."*

This answer **sounds** authoritative. But look closely — the retrieved context says "greenhouse gas emissions," not "CO2 emissions." The aviation and shipping percentages were never in the context at all. The model hallucinated plausible-sounding statistics and subtly shifted the metric from greenhouse gases to CO2.

Without an evaluation framework, this answer passes every manual spot check. With one, we catch it immediately: faithfulness score drops because two of the four claims are unsupported by the retrieved context.

This is why we need rigorous evaluation. Let us build it.

In [ ]:
#@title 🎧 Listen: Why Matter Code
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/05_06_why_matter_code.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

In [ ]:
# Let us see this failure case concretely

failure_example = {
    "question": "What percentage of global CO2 emissions come from transportation?",
    "retrieved_context": (
        "Transportation accounts for approximately 16% of global greenhouse gas "
        "emissions, with road vehicles responsible for nearly three-quarters of "
        "transport emissions."
    ),
    "generated_answer": (
        "Transportation accounts for approximately 16% of global CO2 emissions. "
        "Road vehicles are responsible for about 75% of this, while aviation "
        "contributes roughly 12% and shipping adds about 11%."
    ),
    "ground_truth": (
        "Transportation accounts for about 16% of global greenhouse gas emissions. "
        "Road vehicles are responsible for nearly three-quarters of transport emissions."
    )
}

# Manual claim analysis
claims = [
    ("16% of global CO2 emissions", False, "Context says 'greenhouse gas', not 'CO2'"),
    ("Road vehicles ~75% of transport", True, "Supported: 'nearly three-quarters'"),
    ("Aviation ~12%", False, "Not mentioned in context at all"),
    ("Shipping ~11%", False, "Not mentioned in context at all"),
]

print("=" * 70)
print("  FAILURE CASE: Looks correct, but is unfaithful")
print("=" * 70)
print(f"\nQuestion: {failure_example['question']}")
print(f"\nRetrieved context: {failure_example['retrieved_context']}")
print(f"\nGenerated answer: {failure_example['generated_answer']}")
print(f"\n{'Claim':<45} {'Supported?':<12} {'Reason'}")
print("-" * 90)
for claim, supported, reason in claims:
    status = "YES" if supported else "NO"
    print(f"  {claim:<43} {status:<12} {reason}")

supported_count = sum(1 for _, s, _ in claims if s)
total_claims = len(claims)
faithfulness = supported_count / total_claims

print(f"\nFaithfulness score: {supported_count}/{total_claims} = {faithfulness:.2f}")
print(f"\nThis answer would pass a casual review but FAILS rigorous evaluation.")
print(f"Without metrics, you ship hallucinated statistics to users.")

In [ ]:
#@title 🎧 Listen: Intuition Concept
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/05_07_intuition_concept.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

## 2. Building Intuition

Think of RAG evaluation as an **open-book exam with a grader**. The exam has three dimensions, and the grader checks each one independently.

**Dimension 1 — Did you find the right pages? (Context Relevance)**

You are taking an open-book exam about climate science. You flip through your textbook and bring five pages to your desk. But two of those pages are about ocean currents, one is about volcanic activity, and only two are actually about CO2 emissions. Your retrieval was sloppy — you wasted precious desk space on irrelevant material.

Context relevance asks: *of all the chunks you retrieved, how many are actually relevant to the question?* A perfect retrieval system brings only relevant pages. A poor one wastes context tokens on noise, which can actually confuse the model (the "lost in the middle" phenomenon we discussed in Notebook 3).

**Dimension 2 — Did you only write what was on those pages? (Faithfulness)**

Now you write your answer. A faithful student only includes information from the pages in front of them. An unfaithful student mixes in facts they remember from a lecture two weeks ago, or worse, makes up statistics that sound plausible but are not on any page.

Faithfulness asks: *of all the claims in the generated answer, how many can be traced back to the retrieved context?* This is the hallucination detector. A faithfulness score of 1.0 means every claim is grounded. A score of 0.5 means half the answer is potentially fabricated.

**Dimension 3 — Was your final answer correct? (Answer Correctness)**

Even if you found the right pages and faithfully reported what was on them, your answer might still be wrong — perhaps you misread a number, or the pages themselves were outdated. Answer correctness compares the final answer against a ground-truth reference.

Together, these three metrics form a **diagnostic triangle**. If correctness is low but faithfulness is high, the problem is in retrieval (wrong documents). If faithfulness is low but retrieval is good, the problem is in generation (the model is ignoring or embellishing the context). If all three are low, the entire pipeline needs work.

### Think About This

Before we formalize the mathematics, consider these scenarios and predict which metric fails:

- The retriever returns chunks about the wrong topic, but the model happens to know the answer from training. *Which metric is low? Which is high?*
- The retriever returns perfect chunks, but the model adds extra details from parametric knowledge. *Which metric drops?*
- The retriever returns perfect chunks, the model is faithful to them, but the chunks contain outdated information. *Which metric catches this?*

Think about these for a moment. The answers will become obvious once we write the formal definitions.

In [ ]:
#@title 🎧 Listen: Math Concept
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/05_08_math_concept.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

## 3. The Mathematics

Let us now formalize the three RAGAS metrics. These are not arbitrary scores — each one captures a specific failure mode with a precise mathematical definition.

### 3.1 Faithfulness

Given a generated answer $a$ and retrieved context $C$, faithfulness measures what fraction of the answer's claims are supported by the context.

First, we decompose the answer into individual claims $\{c_1, c_2, \ldots, c_n\}$. Then for each claim, we check whether it is supported by (i.e., can be inferred from) the context:

$$\text{Faithfulness}(a, C) = \frac{|\{c_i : c_i \text{ is supported by } C\}|}{n}$$

A faithfulness of 1.0 means every claim is grounded. A faithfulness of 0.0 means the entire answer is hallucinated.

### 3.2 Context Relevance

Given a question $q$ and retrieved chunks $\{d_1, d_2, \ldots, d_k\}$, context relevance measures what fraction of the retrieved chunks are actually relevant to answering the question:

$$\text{Context Relevance}(q, D) = \frac{|\{d_j : d_j \text{ is relevant to } q\}|}{k}$$

This is essentially the **precision** of retrieval. High precision means we are not wasting context tokens on irrelevant material. Note that this does not measure recall (whether we found *all* relevant chunks) — that is a separate concern.

### 3.3 Answer Correctness

Given a generated answer $a$ and ground-truth answer $a^*$, answer correctness measures how close the generated answer is to the reference. We combine two approaches:

**Token-level F1 score** captures word overlap:

$$\text{Precision} = \frac{|\text{tokens}(a) \cap \text{tokens}(a^*)|}{|\text{tokens}(a)|}$$

$$\text{Recall} = \frac{|\text{tokens}(a) \cap \text{tokens}(a^*)|}{|\text{tokens}(a^*)|}$$

$$\text{F1} = 2 \cdot \frac{\text{Precision} \cdot \text{Recall}}{\text{Precision} + \text{Recall}}$$

**Semantic similarity** captures meaning even when different words are used. We embed both answers and compute cosine similarity:

$$\text{Semantic}(a, a^*) = \frac{E(a) \cdot E(a^*)}{\|E(a)\| \cdot \|E(a^*)\|}$$

The final answer correctness is a weighted combination:

$$\text{Answer Correctness} = \alpha \cdot \text{F1} + (1 - \alpha) \cdot \text{Semantic Similarity}$$

where $\alpha = 0.5$ by default, weighting both components equally.

In [ ]:
#@title 🎧 Listen: Eval Triangle Viz
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/05_09_eval_triangle_viz.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

In [ ]:
# Visualization: The three RAGAS metrics and what failures each catches

fig, ax = plt.subplots(1, 1, figsize=(12, 8))
ax.set_xlim(-3.5, 3.5)
ax.set_ylim(-2.5, 3.5)
ax.set_aspect('equal')
ax.axis('off')
ax.set_title("The RAG Evaluation Triangle: Three Metrics, Three Failure Modes",
             fontsize=14, fontweight='bold', pad=20)

# Draw three overlapping circles (Venn-diagram style)
from matplotlib.patches import Circle, FancyArrowPatch

circle_r = 1.6

# Positions for the three circles (triangle arrangement)
positions = [
    (0, 1.8),     # Top: Faithfulness
    (-1.5, -0.5), # Bottom-left: Context Relevance
    (1.5, -0.5),  # Bottom-right: Answer Correctness
]

colors = ['#E3F2FD', '#E8F5E9', '#FFF3E0']
edge_colors = ['#1565C0', '#2E7D32', '#E65100']
labels = ['Faithfulness', 'Context\nRelevance', 'Answer\nCorrectness']
formulas = [
    'supported claims\n/ total claims',
    'relevant chunks\n/ total chunks',
    'F1 + semantic\nsimilarity'
]
catches = [
    'Catches:\nHallucination',
    'Catches:\nIrrelevant retrieval',
    'Catches:\nWrong answers'
]

for i, (x, y) in enumerate(positions):
    circle = Circle((x, y), circle_r, facecolor=colors[i], edgecolor=edge_colors[i],
                     linewidth=2.5, alpha=0.5)
    ax.add_patch(circle)

    # Label and formula
    ax.text(x, y + 0.35, labels[i], ha='center', va='center',
            fontsize=13, fontweight='bold', color=edge_colors[i])
    ax.text(x, y - 0.2, formulas[i], ha='center', va='center',
            fontsize=9, color=edge_colors[i], style='italic')

    # What it catches (outside the circles)
    offset_x = x * 1.5
    offset_y = y * 1.5 if y > 0 else y * 2.0
    ax.text(offset_x, offset_y, catches[i], ha='center', va='center',
            fontsize=9, fontweight='bold', color=edge_colors[i],
            bbox=dict(boxstyle='round,pad=0.3', facecolor=colors[i],
                      edgecolor=edge_colors[i], alpha=0.8))

# Center label
ax.text(0, 0.4, "Complete\nRAG Quality", ha='center', va='center',
        fontsize=10, fontweight='bold', color='#333',
        bbox=dict(boxstyle='round,pad=0.3', facecolor='white',
                  edgecolor='#666', alpha=0.9))

# Failure mode annotations at edges
failure_modes = [
    (-2.0, 2.8, "Low faithfulness + high correctness:\nModel knows answer but fabricates sources",
     '#1565C0'),
    (-3.2, -1.8, "Low relevance + high faithfulness:\nModel is faithful to irrelevant context",
     '#2E7D32'),
    (3.2, -1.8, "Low correctness + high faithfulness:\nContext itself is wrong or outdated",
     '#E65100'),
]

for x, y, text, color in failure_modes:
    ax.text(x, y, text, ha='center', va='center', fontsize=8,
            color=color, style='italic')

plt.tight_layout()
plt.show()

print("Each metric catches a different failure mode.")
print("All three together give a complete picture of RAG quality.")

In [ ]:
#@title 🎧 Listen: Build It Transition
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/05_10_build_it_transition.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

## 4. Let Us Build It

We will build a complete, evaluated RAG system step by step. First, the knowledge base and test set. Then the full pipeline. Then each evaluation scorer. Everything from scratch.

### 4.1 Setting Up the LLM

We use the Anthropic API with Claude. Store your API key in Google Colab's Secrets panel (key icon in the left sidebar) with the name `ANTHROPIC_API_KEY`.

In [ ]:
#@title 🎧 Listen: Setup Llm
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/05_11_setup_llm.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

In [ ]:
from anthropic import Anthropic
from google.colab import userdata
import os

os.environ['ANTHROPIC_API_KEY'] = userdata.get('ANTHROPIC_API_KEY')
client = Anthropic()

MODEL = "claude-sonnet-4-20250514"

def query_llm(system_prompt: str, user_message: str, max_tokens: int = 1024) -> str:
    """Send a message to Claude and return the response text."""
    response = client.messages.create(
        model=MODEL,
        max_tokens=max_tokens,
        system=system_prompt,
        messages=[{"role": "user", "content": user_message}]
    )
    return response.content[0].text.strip()

def query_llm_json(system_prompt: str, user_message: str, max_tokens: int = 1024) -> dict:
    """Send a message to Claude and parse the response as JSON."""
    raw = query_llm(system_prompt, user_message, max_tokens)
    # Extract JSON from response (handles markdown-wrapped JSON)
    if "```json" in raw:
        raw = raw.split("```json")[1].split("```")[0]
    elif "```" in raw:
        raw = raw.split("```")[1].split("```")[0]
    return json.loads(raw.strip())

# Quick test
test_response = query_llm(
    "You are a helpful assistant.",
    "Say 'RAG evaluation is ready!' in exactly those words."
)
print(f"LLM test: {test_response}")
print(f"Using model: {MODEL}")

In [ ]:
#@title 🎧 Listen: Knowledge Base
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/05_12_knowledge_base.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

### 4.2 The Knowledge Base

We will build a knowledge base about renewable energy — a domain with enough factual detail to make evaluation meaningful. Each paragraph contains specific, verifiable claims.

In [ ]:
# Knowledge base: 24 paragraphs about renewable energy
KNOWLEDGE_BASE = [
    # Solar energy
    "Solar photovoltaic (PV) technology converts sunlight directly into electricity using semiconductor materials, most commonly silicon. When photons strike a PV cell, they knock electrons free from atoms in the semiconductor, creating an electric current. Modern monocrystalline silicon panels achieve commercial efficiencies of 20-22%, while laboratory cells have reached 26.7%. The theoretical maximum efficiency for a single-junction silicon cell, known as the Shockley-Queisser limit, is approximately 33.7%.",

    "Global solar PV capacity reached approximately 1,185 gigawatts (GW) by the end of 2022, with China accounting for roughly 37% of total installations. The cost of solar electricity has declined by about 89% since 2010, making it the cheapest source of new electricity generation in most parts of the world. Utility-scale solar farms can now produce electricity at $20-40 per megawatt-hour (MWh) in favorable locations.",

    "Solar panel degradation is a critical factor in long-term economics. Most manufacturers guarantee that panels will produce at least 80% of their rated output after 25 years. The typical degradation rate is 0.5-0.8% per year. Temperature significantly affects solar panel performance — efficiency drops by approximately 0.3-0.5% for every degree Celsius above 25 degrees C. This is why panels in hot climates often underperform their rated capacity.",

    "Concentrated solar power (CSP) uses mirrors or lenses to concentrate sunlight onto a receiver, which heats a fluid to drive a turbine. Unlike PV, CSP can include thermal energy storage, allowing electricity generation after sunset. The largest CSP plant is the Ivanpah Solar Electric Generating System in California, with a capacity of 392 MW. CSP is generally more expensive than PV but offers the advantage of built-in energy storage.",

    # Wind energy
    "Wind turbines convert the kinetic energy of moving air into electricity. The power available in wind is proportional to the cube of wind speed — doubling wind speed increases available power by a factor of eight. Modern onshore wind turbines typically have capacities of 2-5 MW, with rotor diameters of 100-170 meters. The global average capacity factor for onshore wind is approximately 35%, meaning turbines generate about 35% of their theoretical maximum output over a year.",

    "Offshore wind energy has grown rapidly due to stronger and more consistent winds at sea. Offshore turbines can reach 15 MW or more, with rotor diameters exceeding 220 meters. The global offshore wind capacity was approximately 64 GW at the end of 2022, with the United Kingdom, China, and Germany leading installations. Offshore wind capacity factors typically range from 40-55%, significantly higher than onshore installations.",

    "The levelized cost of energy (LCOE) for onshore wind has declined by approximately 70% since 2010, reaching $30-50 per MWh in favorable locations. Offshore wind costs have also dropped dramatically but remain higher, typically $50-100 per MWh. Wind energy faces challenges including intermittency, noise concerns for onshore installations, and impacts on wildlife, particularly birds and bats.",

    "Wind farm wake effects reduce the energy production of downstream turbines. When wind passes through a turbine, it loses energy and becomes more turbulent. Turbines directly behind the first one can see power reductions of 10-40%. Optimal turbine spacing is typically 5-10 rotor diameters apart. Advanced wake steering techniques, where upstream turbines intentionally yaw to redirect their wake, can recover 5-10% of lost production.",

    # Battery storage
    "Lithium-ion batteries dominate the energy storage market, with costs declining from approximately $1,200 per kilowatt-hour (kWh) in 2010 to around $140 per kWh in 2023. The most common chemistries are lithium nickel manganese cobalt oxide (NMC) for high energy density and lithium iron phosphate (LFP) for longer cycle life and improved safety. LFP batteries can achieve 3,000-5,000 charge cycles before significant degradation.",

    "Grid-scale battery storage is essential for integrating intermittent renewable energy. The largest battery installation is the Moss Landing Energy Storage Facility in California, with a capacity of 750 MW and 3,000 MWh. Battery storage can provide multiple grid services including peak shaving, frequency regulation, and renewable energy time-shifting. The round-trip efficiency of lithium-ion battery systems is typically 85-95%.",

    "Beyond lithium-ion, emerging storage technologies include solid-state batteries, flow batteries, and compressed air energy storage. Sodium-ion batteries are a promising alternative that avoids scarce lithium and cobalt. Iron-air batteries, being developed by Form Energy, promise 100-hour storage durations at costs below $20 per kWh, which would be transformative for seasonal energy storage.",

    "Battery degradation follows a non-linear pattern. Capacity loss is typically fastest in the first few hundred cycles, then slows. Operating temperature significantly affects battery lifespan — extreme heat or cold accelerates degradation. Most grid-scale battery warranties guarantee 70-80% capacity retention after 10-15 years. Calendar aging (degradation over time regardless of use) contributes roughly 1-2% capacity loss per year.",

    # Hydrogen
    "Green hydrogen is produced by electrolysis of water using renewable electricity. The process splits water molecules into hydrogen and oxygen, with the hydrogen stored for later use. Current electrolyzers achieve efficiencies of 60-80%, meaning 60-80% of the input electricity is converted to hydrogen energy. The cost of green hydrogen was approximately $4-8 per kilogram in 2023, compared to $1-2 per kg for grey hydrogen produced from natural gas.",

    "Hydrogen can be used as an energy carrier for long-duration storage, industrial processes, and transportation. In fuel cells, hydrogen combines with oxygen to produce electricity, water, and heat, with electrical efficiencies of 40-60%. The round-trip efficiency of hydrogen (electricity to hydrogen and back to electricity) is only about 25-35%, significantly lower than battery storage, making it more suitable for applications where batteries are impractical.",

    "Blue hydrogen is produced from natural gas with carbon capture and storage (CCS), capturing approximately 85-95% of the CO2 emissions. However, upstream methane leakage during natural gas extraction and transport can significantly reduce the climate benefits. A 2021 study by Howarth and Jacobson found that blue hydrogen may have a larger greenhouse gas footprint than simply burning natural gas, when accounting for methane leakage rates above 3.5%.",

    # Grid integration
    "Integrating high shares of renewable energy into the electrical grid requires addressing variability and uncertainty. The duck curve, first identified by the California Independent System Operator (CAISO), describes the phenomenon where midday solar generation creates a steep ramp in net demand during evening hours. In California, the evening ramp can exceed 13 GW in just three hours, requiring fast-responding generation sources.",

    "Grid frequency must be maintained at exactly 50 Hz (in Europe) or 60 Hz (in North America) for stable operation. Traditional synchronous generators inherently provide inertia that helps stabilize frequency. As inverter-based renewables (solar and wind) replace synchronous generators, the grid loses inertia. Grid-forming inverters, a newer technology, can synthetically provide inertia by programming inverters to mimic the behavior of synchronous machines.",

    "Demand response programs incentivize electricity consumers to shift their usage to times of high renewable generation. Smart thermostats, electric vehicle charging, and industrial processes can be scheduled to absorb excess solar generation during midday hours. Studies estimate that demand response could reduce peak demand by 10-20% in most electrical grids, significantly reducing the need for peaking power plants.",

    # Policy and economics
    "The global investment in renewable energy reached approximately $495 billion in 2022, with solar accounting for the largest share at about $308 billion. China led global investment at approximately $168 billion, followed by the United States at $73 billion and the European Union at $60 billion. Despite record investment, the International Energy Agency (IEA) estimates that annual clean energy investment needs to reach $4 trillion by 2030 to achieve net-zero emissions by 2050.",

    "Feed-in tariffs (FITs) were among the earliest and most effective policy mechanisms for promoting renewable energy adoption. Under a FIT, renewable energy producers are guaranteed a fixed price for their electricity over a long-term contract (typically 15-25 years). Germany's Erneuerbare-Energien-Gesetz (EEG), introduced in 2000, is credited with driving the early growth of both solar and wind energy in Europe.",

    "Carbon pricing mechanisms include carbon taxes and cap-and-trade systems. The European Union Emissions Trading System (EU ETS), the world's largest carbon market, covers approximately 40% of EU greenhouse gas emissions. Carbon prices in the EU ETS have risen from under 5 euros per tonne in 2017 to over 80 euros per tonne in 2023. Economists generally estimate that a carbon price of $50-100 per tonne of CO2 is needed to drive sufficient decarbonization.",

    # Environmental considerations
    "The lifecycle greenhouse gas emissions of renewable energy sources are significantly lower than fossil fuels. Solar PV emits approximately 20-50 grams of CO2-equivalent per kWh over its lifecycle (including manufacturing and installation), compared to 410-650 g CO2eq/kWh for natural gas and 740-1,200 g CO2eq/kWh for coal. Wind energy has even lower lifecycle emissions at 7-15 g CO2eq/kWh.",

    "End-of-life management for renewable energy equipment is an emerging challenge. An estimated 78 million tonnes of solar panel waste will be generated globally by 2050. Current recycling processes can recover about 80-90% of the glass and metals in a solar panel. Wind turbine blades, typically made from fiberglass composites, are particularly difficult to recycle and many end up in landfills. New blade designs using thermoplastic resins may enable easier recycling in the future.",

    "Land use is a significant consideration for renewable energy deployment. Utility-scale solar farms require approximately 5-10 acres per MW of capacity. Onshore wind farms require roughly 30-60 acres per MW, though most of the land between turbines can still be used for agriculture. Agrivoltaics — the co-location of solar panels and farming — is an emerging approach that can increase total land productivity by 60-70% compared to using the land for either purpose alone.",
]

print(f"Knowledge base: {len(KNOWLEDGE_BASE)} paragraphs")
print(f"Total characters: {sum(len(p) for p in KNOWLEDGE_BASE):,}")
print(f"Average paragraph length: {np.mean([len(p) for p in KNOWLEDGE_BASE]):.0f} chars")
print(f"\nTopics covered:")
topics = ["Solar Energy (4)", "Wind Energy (4)", "Battery Storage (4)",
          "Hydrogen (3)", "Grid Integration (3)", "Policy & Economics (3)",
          "Environmental (3)"]
for t in topics:
    print(f"  - {t}")

In [ ]:
#@title 🎧 Listen: Test Set
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/05_13_test_set.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

### 4.3 The Test Set

A good evaluation requires ground-truth question-answer pairs. We create 10 questions spanning different difficulty levels and knowledge areas.

In [ ]:
# Test set: 10 questions with ground truth answers
TEST_SET = [
    {
        "question": "What is the theoretical maximum efficiency for a single-junction silicon solar cell?",
        "ground_truth": "The theoretical maximum efficiency for a single-junction silicon solar cell, known as the Shockley-Queisser limit, is approximately 33.7%.",
        "difficulty": "easy",
        "topic": "solar"
    },
    {
        "question": "How much has the cost of solar electricity declined since 2010?",
        "ground_truth": "The cost of solar electricity has declined by about 89% since 2010, making it the cheapest source of new electricity generation in most parts of the world.",
        "difficulty": "easy",
        "topic": "solar"
    },
    {
        "question": "What is the relationship between wind speed and available wind power?",
        "ground_truth": "The power available in wind is proportional to the cube of wind speed. Doubling wind speed increases available power by a factor of eight.",
        "difficulty": "medium",
        "topic": "wind"
    },
    {
        "question": "How do offshore wind capacity factors compare to onshore?",
        "ground_truth": "Offshore wind capacity factors typically range from 40-55%, significantly higher than onshore installations which have a global average capacity factor of approximately 35%.",
        "difficulty": "medium",
        "topic": "wind"
    },
    {
        "question": "What is the round-trip efficiency of hydrogen energy storage compared to batteries?",
        "ground_truth": "The round-trip efficiency of hydrogen (electricity to hydrogen and back to electricity) is only about 25-35%, significantly lower than lithium-ion battery storage which achieves 85-95% round-trip efficiency.",
        "difficulty": "hard",
        "topic": "hydrogen_vs_battery"
    },
    {
        "question": "What is the duck curve and why is it a problem?",
        "ground_truth": "The duck curve, first identified by CAISO, describes the phenomenon where midday solar generation creates a steep ramp in net demand during evening hours. In California, the evening ramp can exceed 13 GW in just three hours, requiring fast-responding generation sources.",
        "difficulty": "medium",
        "topic": "grid"
    },
    {
        "question": "How much annual clean energy investment is needed to reach net-zero by 2050?",
        "ground_truth": "The International Energy Agency (IEA) estimates that annual clean energy investment needs to reach $4 trillion by 2030 to achieve net-zero emissions by 2050. Global investment in 2022 was approximately $495 billion.",
        "difficulty": "medium",
        "topic": "policy"
    },
    {
        "question": "What are the lifecycle greenhouse gas emissions of solar PV compared to coal?",
        "ground_truth": "Solar PV emits approximately 20-50 grams of CO2-equivalent per kWh over its lifecycle, compared to 740-1,200 g CO2eq/kWh for coal. Wind energy has even lower emissions at 7-15 g CO2eq/kWh.",
        "difficulty": "medium",
        "topic": "environmental"
    },
    {
        "question": "What problem does blue hydrogen face regarding methane leakage?",
        "ground_truth": "A 2021 study by Howarth and Jacobson found that blue hydrogen may have a larger greenhouse gas footprint than simply burning natural gas, when accounting for methane leakage rates above 3.5% during natural gas extraction and transport.",
        "difficulty": "hard",
        "topic": "hydrogen"
    },
    {
        "question": "How can wake effects in wind farms be mitigated?",
        "ground_truth": "Wind farm wake effects reduce downstream turbine power by 10-40%. Optimal turbine spacing of 5-10 rotor diameters helps. Advanced wake steering techniques, where upstream turbines intentionally yaw to redirect their wake, can recover 5-10% of lost production.",
        "difficulty": "hard",
        "topic": "wind"
    },
]

print(f"Test set: {len(TEST_SET)} questions")
print(f"\nDifficulty distribution:")
for diff in ["easy", "medium", "hard"]:
    count = sum(1 for q in TEST_SET if q["difficulty"] == diff)
    print(f"  {diff}: {count}")
print(f"\nSample question: \"{TEST_SET[0]['question']}\"")
print(f"Ground truth: \"{TEST_SET[0]['ground_truth'][:80]}...\"")

In [ ]:
#@title 🎧 Listen: Rag Pipeline Intro
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/05_14_rag_pipeline_intro.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

### 4.4 The Complete RAG Pipeline

Now we build the full pipeline: chunking, embedding, indexing, retrieval, and generation. We wrap everything in a clean class so we can easily swap components for the ablation study later.

In [ ]:
#@title 🎧 Listen: Load Embedding Model
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/05_15_load_embedding_model.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

In [ ]:
from sentence_transformers import SentenceTransformer
import faiss

# Load the embedding model
embedding_model = SentenceTransformer('all-MiniLM-L6-v2')
EMBEDDING_DIM = 384

print(f"Embedding model loaded: all-MiniLM-L6-v2 ({EMBEDDING_DIM} dimensions)")

In [ ]:
#@title 🎧 Listen: Rag Pipeline Class
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/05_16_rag_pipeline_class.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

In [ ]:
class RAGPipeline:
    """
    A complete RAG pipeline with configurable components.
    Supports fixed and semantic chunking, dense and hybrid retrieval, and optional reranking.
    """

    def __init__(self, knowledge_base: List[str], config: Dict = None):
        self.knowledge_base = knowledge_base
        self.config = config or {}

        # Configuration with defaults
        self.chunk_size = self.config.get('chunk_size', 200)
        self.chunk_overlap = self.config.get('chunk_overlap', 30)
        self.chunking_method = self.config.get('chunking_method', 'fixed')
        self.retrieval_method = self.config.get('retrieval_method', 'dense')
        self.use_reranking = self.config.get('use_reranking', False)
        self.top_k = self.config.get('top_k', 3)

        # State
        self.chunks = []
        self.chunk_embeddings = None
        self.faiss_index = None
        self.bm25_index = None

        # Build the pipeline
        self._chunk_documents()
        self._build_indices()

    def _chunk_documents(self):
        """Chunk the knowledge base using the configured method."""
        if self.chunking_method == 'semantic':
            self._semantic_chunk()
        else:
            self._fixed_chunk()

    def _fixed_chunk(self):
        """Fixed-size chunking with word-level splitting and overlap."""
        self.chunks = []
        for doc in self.knowledge_base:
            words = doc.split()
            start = 0
            while start < len(words):
                end = min(start + self.chunk_size, len(words))
                chunk_text = ' '.join(words[start:end])
                self.chunks.append({
                    'text': chunk_text,
                    'source_idx': self.knowledge_base.index(doc)
                })
                if end >= len(words):
                    break
                start = end - self.chunk_overlap

    def _semantic_chunk(self):
        """Semantic chunking: split at sentence boundaries, group by similarity."""
        self.chunks = []
        for doc_idx, doc in enumerate(self.knowledge_base):
            sentences = re.split(r'(?<=[.!?])\s+', doc)
            if len(sentences) <= 2:
                self.chunks.append({'text': doc, 'source_idx': doc_idx})
                continue

            # Embed sentences
            sent_embeddings = embedding_model.encode(sentences, normalize_embeddings=True)

            current_chunk = [sentences[0]]
            for i in range(1, len(sentences)):
                sim = np.dot(sent_embeddings[i-1], sent_embeddings[i])
                if sim < 0.5 and len(' '.join(current_chunk).split()) > 30:
                    self.chunks.append({
                        'text': ' '.join(current_chunk),
                        'source_idx': doc_idx
                    })
                    current_chunk = [sentences[i]]
                else:
                    current_chunk.append(sentences[i])

            if current_chunk:
                self.chunks.append({
                    'text': ' '.join(current_chunk),
                    'source_idx': doc_idx
                })

    def _build_indices(self):
        """Build FAISS (dense) and optionally BM25 (sparse) indices."""
        # Dense index
        chunk_texts = [c['text'] for c in self.chunks]
        self.chunk_embeddings = embedding_model.encode(
            chunk_texts, normalize_embeddings=True
        ).astype('float32')

        self.faiss_index = faiss.IndexFlatIP(EMBEDDING_DIM)
        self.faiss_index.add(self.chunk_embeddings)

        # BM25 index (for hybrid retrieval)
        if self.retrieval_method == 'hybrid':
            from rank_bm25 import BM25Okapi
            tokenized = [text.lower().split() for text in chunk_texts]
            self.bm25_index = BM25Okapi(tokenized)

    def retrieve(self, query: str) -> List[Dict]:
        """Retrieve the most relevant chunks for a query."""
        if self.retrieval_method == 'hybrid':
            return self._hybrid_retrieve(query)
        else:
            return self._dense_retrieve(query)

    def _dense_retrieve(self, query: str) -> List[Dict]:
        """Pure dense retrieval using FAISS."""
        query_embedding = embedding_model.encode(
            [query], normalize_embeddings=True
        ).astype('float32')

        k = self.top_k * 3 if self.use_reranking else self.top_k
        distances, indices = self.faiss_index.search(query_embedding, k)

        results = []
        for score, idx in zip(distances[0], indices[0]):
            if idx < len(self.chunks):
                results.append({
                    'text': self.chunks[idx]['text'],
                    'score': float(score),
                    'source_idx': self.chunks[idx]['source_idx'],
                    'chunk_idx': int(idx)
                })
        return results[:k]

    def _hybrid_retrieve(self, query: str) -> List[Dict]:
        """Hybrid retrieval: combine dense and BM25 with reciprocal rank fusion."""
        # Dense results
        query_embedding = embedding_model.encode(
            [query], normalize_embeddings=True
        ).astype('float32')
        k_retrieve = self.top_k * 3 if self.use_reranking else self.top_k
        k_each = k_retrieve * 2

        distances, indices = self.faiss_index.search(query_embedding, k_each)

        # BM25 results
        tokenized_query = query.lower().split()
        bm25_scores = self.bm25_index.get_scores(tokenized_query)
        bm25_top = np.argsort(bm25_scores)[::-1][:k_each]

        # Reciprocal Rank Fusion
        rrf_scores = {}
        rrf_k = 60  # Standard RRF constant

        for rank, idx in enumerate(indices[0]):
            idx = int(idx)
            if idx < len(self.chunks):
                rrf_scores[idx] = rrf_scores.get(idx, 0) + 1.0 / (rank + rrf_k)

        for rank, idx in enumerate(bm25_top):
            idx = int(idx)
            rrf_scores[idx] = rrf_scores.get(idx, 0) + 1.0 / (rank + rrf_k)

        # Sort by fused score
        ranked = sorted(rrf_scores.items(), key=lambda x: x[1], reverse=True)

        results = []
        for idx, score in ranked[:k_retrieve]:
            results.append({
                'text': self.chunks[idx]['text'],
                'score': float(score),
                'source_idx': self.chunks[idx]['source_idx'],
                'chunk_idx': idx
            })
        return results

    def _rerank(self, query: str, candidates: List[Dict]) -> List[Dict]:
        """Rerank candidates using LLM-as-judge scoring."""
        if not self.use_reranking or len(candidates) <= self.top_k:
            return candidates[:self.top_k]

        rerank_prompt = f"""Rate how relevant each document is to answering this question.
Question: {query}

For each document, respond with a relevance score from 0 to 10.
Return ONLY a JSON list of scores, e.g., [8, 3, 9, ...]"""

        docs_text = "\n\n".join(
            f"Document {i+1}: {c['text'][:300]}"
            for i, c in enumerate(candidates)
        )

        try:
            result = query_llm("You are a relevance scoring system.",
                             rerank_prompt + "\n\n" + docs_text, max_tokens=256)
            # Parse scores
            scores_match = re.search(r'\[[\d,\s.]+\]', result)
            if scores_match:
                scores = json.loads(scores_match.group())
                for i, score in enumerate(scores):
                    if i < len(candidates):
                        candidates[i]['rerank_score'] = float(score)
                candidates.sort(key=lambda x: x.get('rerank_score', 0), reverse=True)
        except Exception as e:
            pass  # Fall back to original ordering

        return candidates[:self.top_k]

    def generate(self, query: str, retrieved_chunks: List[Dict]) -> str:
        """Generate an answer using the retrieved context."""
        context = "\n\n---\n\n".join(
            f"[Source {i+1}]: {chunk['text']}"
            for i, chunk in enumerate(retrieved_chunks)
        )

        system_prompt = """You are a knowledgeable assistant that answers questions based ONLY on the provided context.
Rules:
- Only use information from the provided sources
- If the context does not contain enough information, say so clearly
- Be specific and cite numbers/data when available
- Keep answers concise but complete (2-4 sentences)"""

        user_message = f"""Context:
{context}

Question: {query}

Answer based ONLY on the context above:"""

        return query_llm(system_prompt, user_message, max_tokens=512)

    def __call__(self, query: str) -> Tuple[str, List[Dict]]:
        """Run the full pipeline: retrieve → (rerank) → generate."""
        candidates = self.retrieve(query)
        if self.use_reranking:
            retrieved = self._rerank(query, candidates)
        else:
            retrieved = candidates[:self.top_k]
        answer = self.generate(query, retrieved)
        return answer, retrieved

    def get_stats(self) -> Dict:
        """Return pipeline statistics."""
        return {
            'num_chunks': len(self.chunks),
            'chunking_method': self.chunking_method,
            'retrieval_method': self.retrieval_method,
            'use_reranking': self.use_reranking,
            'top_k': self.top_k,
            'avg_chunk_length': np.mean([len(c['text'].split()) for c in self.chunks]),
        }

In [ ]:
#@title 🎧 Listen: Build Baseline
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/05_17_build_baseline.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

In [ ]:
# Build the baseline pipeline
baseline_config = {
    'chunk_size': 200,
    'chunk_overlap': 30,
    'chunking_method': 'fixed',
    'retrieval_method': 'dense',
    'use_reranking': False,
    'top_k': 3,
}

pipeline = RAGPipeline(KNOWLEDGE_BASE, config=baseline_config)

stats = pipeline.get_stats()
print("Baseline RAG Pipeline Built")
print("=" * 50)
for key, value in stats.items():
    print(f"  {key}: {value}")

# Quick test
test_answer, test_sources = pipeline(TEST_SET[0]['question'])
print(f"\nTest query: \"{TEST_SET[0]['question']}\"")
print(f"Answer: {test_answer}")
print(f"Sources retrieved: {len(test_sources)}")

In [ ]:
#@title 🎧 Listen: Eval Scorers Intro
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/05_18_eval_scorers_intro.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

### 4.5 Implementing the Evaluation Scorers

Now the heart of this capstone — the three RAGAS-style evaluation metrics, each implemented from scratch using LLM-as-judge.

In [ ]:
#@title 🎧 Listen: Rag Evaluator Class
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/05_19_rag_evaluator_class.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

In [ ]:
class RAGEvaluator:
    """
    RAGAS-style evaluation framework for RAG pipelines.
    Implements faithfulness, context relevance, and answer correctness scorers.
    """

    def __init__(self, embedding_model):
        self.embedding_model = embedding_model

    # ----------------------------------------------------------------
    # Metric 1: Faithfulness
    # ----------------------------------------------------------------
    def score_faithfulness(self, answer: str, context: List[str]) -> Dict:
        """
        Measure what fraction of claims in the answer are supported by the context.
        Uses LLM-as-judge to decompose the answer into claims and verify each one.
        """
        # Step 1: Decompose the answer into individual claims
        decompose_prompt = f"""Break the following answer into individual factual claims.
Each claim should be a single, verifiable statement.

Answer: {answer}

Return a JSON list of claim strings. Example: ["Claim 1 text", "Claim 2 text"]
Return ONLY the JSON list, nothing else."""

        try:
            raw = query_llm("You extract factual claims from text.", decompose_prompt)
            if "```json" in raw:
                raw = raw.split("```json")[1].split("```")[0]
            elif "```" in raw:
                raw = raw.split("```")[1].split("```")[0]
            claims = json.loads(raw.strip())
        except Exception:
            # Fallback: split answer into sentences as claims
            claims = [s.strip() for s in re.split(r'(?<=[.!?])\s+', answer) if len(s.strip()) > 10]

        if not claims:
            return {'score': 0.0, 'claims': [], 'supported': [], 'details': []}

        # Step 2: Check each claim against the context
        context_text = "\n\n".join(context)
        supported = []
        details = []

        for claim in claims:
            verify_prompt = f"""Determine if the following claim is supported by the context.

Context:
{context_text}

Claim: {claim}

Respond with ONLY a JSON object: {{"supported": true/false, "reason": "brief explanation"}}"""

            try:
                raw = query_llm("You verify claims against source context.", verify_prompt, max_tokens=256)
                if "```json" in raw:
                    raw = raw.split("```json")[1].split("```")[0]
                elif "```" in raw:
                    raw = raw.split("```")[1].split("```")[0]
                result = json.loads(raw.strip())
                is_supported = result.get('supported', False)
                reason = result.get('reason', '')
            except Exception:
                is_supported = False
                reason = "Could not verify"

            supported.append(is_supported)
            details.append({'claim': claim, 'supported': is_supported, 'reason': reason})

        score = sum(supported) / len(supported) if supported else 0.0
        return {
            'score': score,
            'claims': claims,
            'supported': supported,
            'details': details,
            'num_claims': len(claims),
            'num_supported': sum(supported)
        }

    # ----------------------------------------------------------------
    # Metric 2: Context Relevance
    # ----------------------------------------------------------------
    def score_context_relevance(self, question: str, retrieved_chunks: List[str]) -> Dict:
        """
        Measure what fraction of retrieved chunks are relevant to the question.
        Uses LLM-as-judge to rate each chunk's relevance.
        """
        relevance_results = []

        for i, chunk in enumerate(retrieved_chunks):
            relevance_prompt = f"""Is the following document relevant to answering the question?
A document is relevant if it contains information that would help answer the question.

Question: {question}
Document: {chunk}

Respond with ONLY a JSON object: {{"relevant": true/false, "reason": "brief explanation"}}"""

            try:
                raw = query_llm("You judge document relevance.", relevance_prompt, max_tokens=256)
                if "```json" in raw:
                    raw = raw.split("```json")[1].split("```")[0]
                elif "```" in raw:
                    raw = raw.split("```")[1].split("```")[0]
                result = json.loads(raw.strip())
                is_relevant = result.get('relevant', False)
                reason = result.get('reason', '')
            except Exception:
                is_relevant = False
                reason = "Could not assess"

            relevance_results.append({
                'chunk_idx': i,
                'relevant': is_relevant,
                'reason': reason,
                'chunk_preview': chunk[:100] + '...'
            })

        relevant_count = sum(1 for r in relevance_results if r['relevant'])
        total = len(relevance_results)
        score = relevant_count / total if total > 0 else 0.0

        return {
            'score': score,
            'num_relevant': relevant_count,
            'num_total': total,
            'details': relevance_results
        }

    # ----------------------------------------------------------------
    # Metric 3: Answer Correctness
    # ----------------------------------------------------------------
    def score_answer_correctness(self, answer: str, ground_truth: str, alpha: float = 0.5) -> Dict:
        """
        Measure answer correctness using F1 token overlap + semantic similarity.
        alpha controls the weighting: alpha * F1 + (1-alpha) * semantic.
        """
        # Component 1: Token-level F1
        answer_tokens = set(answer.lower().split())
        truth_tokens = set(ground_truth.lower().split())

        if len(answer_tokens) == 0 or len(truth_tokens) == 0:
            f1 = 0.0
            precision = 0.0
            recall = 0.0
        else:
            common = answer_tokens & truth_tokens
            precision = len(common) / len(answer_tokens)
            recall = len(common) / len(truth_tokens)
            f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0.0

        # Component 2: Semantic similarity
        embeddings = self.embedding_model.encode(
            [answer, ground_truth], normalize_embeddings=True
        )
        semantic_sim = float(np.dot(embeddings[0], embeddings[1]))

        # Combined score
        combined = alpha * f1 + (1 - alpha) * semantic_sim

        return {
            'score': combined,
            'f1': f1,
            'precision': precision,
            'recall': recall,
            'semantic_similarity': semantic_sim,
            'alpha': alpha
        }

    # ----------------------------------------------------------------
    # Full evaluation
    # ----------------------------------------------------------------
    def evaluate(self, question: str, answer: str, ground_truth: str,
                 retrieved_chunks: List[str]) -> Dict:
        """Run all three metrics on a single question-answer pair."""
        context_texts = retrieved_chunks

        faithfulness = self.score_faithfulness(answer, context_texts)
        context_relevance = self.score_context_relevance(question, context_texts)
        answer_correctness = self.score_answer_correctness(answer, ground_truth)

        return {
            'faithfulness': faithfulness,
            'context_relevance': context_relevance,
            'answer_correctness': answer_correctness,
            'summary': {
                'faithfulness': faithfulness['score'],
                'context_relevance': context_relevance['score'],
                'answer_correctness': answer_correctness['score'],
            }
        }

evaluator = RAGEvaluator(embedding_model)
print("RAG Evaluator initialized with all three RAGAS-style metrics.")

In [ ]:
#@title 🎧 Listen: Run Eval Intro
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/05_20_run_eval_intro.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

### 4.6 Running the Full Evaluation

Let us now run the pipeline on all 10 test questions and evaluate each one.

In [ ]:
#@title 🎧 Listen: Run Eval Code
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/05_21_run_eval_code.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

In [ ]:
# Evaluate all 10 test questions
print("Running full evaluation on 10 test questions...")
print("This will make multiple LLM calls per question — please be patient.\n")

all_results = []

for i, test_case in enumerate(TEST_SET):
    print(f"[{i+1}/10] Evaluating: \"{test_case['question'][:60]}...\"")

    # Run the pipeline
    answer, retrieved = pipeline(test_case['question'])
    retrieved_texts = [r['text'] for r in retrieved]

    # Evaluate
    eval_result = evaluator.evaluate(
        question=test_case['question'],
        answer=answer,
        ground_truth=test_case['ground_truth'],
        retrieved_chunks=retrieved_texts
    )

    all_results.append({
        'question': test_case['question'],
        'difficulty': test_case['difficulty'],
        'topic': test_case['topic'],
        'answer': answer,
        'ground_truth': test_case['ground_truth'],
        'retrieved_texts': retrieved_texts,
        'eval': eval_result
    })

    scores = eval_result['summary']
    print(f"  Faithfulness: {scores['faithfulness']:.2f}  |  "
          f"Relevance: {scores['context_relevance']:.2f}  |  "
          f"Correctness: {scores['answer_correctness']:.2f}")

print("\nEvaluation complete!")

In [ ]:
#@title 🎧 Listen: Per Question Viz
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/05_22_per_question_viz.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

In [ ]:
# Visualization: Per-question breakdown of all 3 metrics as a grouped bar chart

questions_short = [
    textwrap.fill(r['question'][:50] + '...', width=25)
    for r in all_results
]
faithfulness_scores = [r['eval']['summary']['faithfulness'] for r in all_results]
relevance_scores = [r['eval']['summary']['context_relevance'] for r in all_results]
correctness_scores = [r['eval']['summary']['answer_correctness'] for r in all_results]

x = np.arange(len(all_results))
width = 0.25

fig, ax = plt.subplots(1, 1, figsize=(16, 7))

bars1 = ax.bar(x - width, faithfulness_scores, width, label='Faithfulness',
               color='#1565C0', alpha=0.85, edgecolor='white', linewidth=0.5)
bars2 = ax.bar(x, relevance_scores, width, label='Context Relevance',
               color='#2E7D32', alpha=0.85, edgecolor='white', linewidth=0.5)
bars3 = ax.bar(x + width, correctness_scores, width, label='Answer Correctness',
               color='#E65100', alpha=0.85, edgecolor='white', linewidth=0.5)

ax.set_xlabel('Test Questions', fontsize=12)
ax.set_ylabel('Score (0-1)', fontsize=12)
ax.set_title('RAG Evaluation: Per-Question Metric Breakdown (Baseline Pipeline)',
             fontsize=14, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels([f"Q{i+1}" for i in range(len(all_results))], fontsize=10)
ax.set_ylim(0, 1.15)
ax.legend(fontsize=11, loc='upper right')
ax.axhline(y=0.8, color='gray', linestyle='--', alpha=0.5, label='Target (0.8)')

# Add value labels on bars
for bars in [bars1, bars2, bars3]:
    for bar in bars:
        height = bar.get_height()
        ax.text(bar.get_x() + bar.get_width()/2., height + 0.02,
                f'{height:.2f}', ha='center', va='bottom', fontsize=7)

# Add question labels below
for i, q in enumerate(all_results):
    ax.text(i, -0.12, f"{q['difficulty']}", ha='center', va='top',
            fontsize=8, style='italic', color='#666')

plt.tight_layout()
plt.show()

# Print aggregate statistics
avg_faith = np.mean(faithfulness_scores)
avg_rel = np.mean(relevance_scores)
avg_corr = np.mean(correctness_scores)
print(f"\nAggregate Scores:")
print(f"  Faithfulness:       {avg_faith:.3f}")
print(f"  Context Relevance:  {avg_rel:.3f}")
print(f"  Answer Correctness: {avg_corr:.3f}")
print(f"  Overall (average):  {(avg_faith + avg_rel + avg_corr) / 3:.3f}")

In [ ]:
#@title 🎧 Listen: Todo1 Intro
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/05_23_todo1_intro.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

## 5. Your Turn

### TODO 1: Implement the Faithfulness Scorer from Scratch

The evaluator above uses LLM-as-judge for faithfulness. But what if you wanted a **rule-based** faithfulness scorer that does not require an LLM? This is useful for fast iteration during development.

Your task: implement a faithfulness scorer that breaks the answer into claims (sentences) and checks each claim against the context using embedding similarity. If a claim's maximum similarity to any context chunk is below a threshold, it is flagged as unsupported.

In [ ]:
#@title 🎧 Listen: Todo1 Code
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/05_24_todo1_code.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

In [ ]:
def rule_based_faithfulness(answer: str, context_chunks: List[str],
                            threshold: float = 0.5) -> Dict:
    """
    TODO: Implement a rule-based faithfulness scorer.

    Steps:
    1. Split the answer into individual claims (sentences)
    2. Embed each claim and each context chunk
    3. For each claim, compute its maximum cosine similarity to any context chunk
    4. A claim is "supported" if max_similarity >= threshold
    5. Faithfulness = number of supported claims / total claims

    Args:
        answer: The generated answer text
        context_chunks: List of retrieved context strings
        threshold: Similarity threshold for considering a claim supported

    Returns:
        Dict with keys: 'score', 'claims', 'similarities', 'supported'
    """
    # ============ TODO ============
    # Step 1: Split answer into claims (sentences)
    claims = []  # YOUR CODE HERE

    # Step 2: Embed claims and context chunks
    # claim_embeddings = ???
    # context_embeddings = ???

    # Step 3: Compute max similarity for each claim
    similarities = []  # YOUR CODE HERE

    # Step 4: Determine which claims are supported
    supported = []  # YOUR CODE HERE

    # Step 5: Compute final score
    score = 0.0  # YOUR CODE HERE

    return {
        'score': score,
        'claims': claims,
        'similarities': similarities,
        'supported': supported,
    }

    # ============ END TODO ============

# Test your implementation:
# test_answer = "Solar panels achieve 22% efficiency. They work best in cold weather."
# test_context = [
#     "Modern monocrystalline silicon panels achieve commercial efficiencies of 20-22%.",
#     "Temperature significantly affects solar panel performance — efficiency drops by "
#     "approximately 0.3-0.5% for every degree Celsius above 25 degrees C."
# ]
# result = rule_based_faithfulness(test_answer, test_context)
# print(f"Faithfulness: {result['score']:.2f}")
# for claim, sim, sup in zip(result['claims'], result['similarities'], result['supported']):
#     print(f"  [{'+' if sup else '-'}] (sim={sim:.3f}) {claim}")

In [ ]:
#@title 🎧 Listen: Todo2 Intro
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/05_25_todo2_intro.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

### TODO 2: Implement a Retrieval Diagnostic Function

When a RAG system gives a wrong answer, the failure can happen at three different points. Your task: build a diagnostic function that identifies which failure mode is responsible.

**Failure Mode 1 — Wrong Retrieval:** The retrieved chunks are irrelevant to the question. The model never had a chance because it was given the wrong information.

**Failure Mode 2 — Ignored Context:** The retrieved chunks contain the right information, but the model ignored them and generated from parametric knowledge instead.

**Failure Mode 3 — Unfaithful Generation:** The model used the context but added claims that are not in it — hallucinated details blended with real ones.

In [ ]:
#@title 🎧 Listen: Todo2 Code
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/05_26_todo2_code.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

In [ ]:
def diagnose_failure(question: str, answer: str, ground_truth: str,
                     retrieved_chunks: List[str],
                     faithfulness_score: float,
                     relevance_score: float,
                     correctness_score: float) -> Dict:
    """
    TODO: Diagnose which failure mode is responsible for a bad RAG answer.

    Decision logic:
    - If relevance_score < 0.5: Failure Mode 1 (wrong retrieval)
    - If relevance_score >= 0.5 and faithfulness_score < 0.5:
        Could be Mode 2 (ignored context) or Mode 3 (unfaithful generation)
        Check if the answer overlaps with the context at all:
        - If answer has LOW overlap with context: Mode 2 (ignored context)
        - If answer has SOME overlap but adds extra claims: Mode 3 (unfaithful)
    - If all scores are high (>= 0.7): No failure — pipeline is working

    Args:
        question, answer, ground_truth: The Q&A pair
        retrieved_chunks: List of retrieved context strings
        faithfulness_score, relevance_score, correctness_score: The three RAGAS scores

    Returns:
        Dict with: 'failure_mode' (str), 'explanation' (str), 'recommendation' (str)
    """
    # ============ TODO ============

    failure_mode = "unknown"
    explanation = ""
    recommendation = ""

    # YOUR DECISION LOGIC HERE
    # Consider edge cases:
    # - What if correctness is high but faithfulness is low?
    #   (Model knows the answer from training but is not grounded in context)
    # - What if faithfulness is high but correctness is low?
    #   (Model is faithful to wrong/outdated context)

    return {
        'failure_mode': failure_mode,
        'explanation': explanation,
        'recommendation': recommendation,
    }

    # ============ END TODO ============

# Test your implementation:
# diagnosis = diagnose_failure(
#     question="What is the efficiency of solar panels?",
#     answer="Solar panels are about 50% efficient.",
#     ground_truth="Modern monocrystalline silicon panels achieve 20-22% efficiency.",
#     retrieved_chunks=["Wind turbines convert kinetic energy of air into electricity."],
#     faithfulness_score=0.0,
#     relevance_score=0.0,
#     correctness_score=0.15
# )
# print(f"Failure mode: {diagnosis['failure_mode']}")
# print(f"Explanation: {diagnosis['explanation']}")
# print(f"Recommendation: {diagnosis['recommendation']}")

In [ ]:
#@title 🎧 Listen: Ablation Intro
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/05_27_ablation_intro.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

## 6. Pipeline Ablation Study

Now we compare four RAG configurations to see how each improvement affects our three metrics. This is how you make evidence-based decisions about pipeline design.

The four configurations:
1. **Baseline**: Fixed chunking + dense retrieval (no reranking)
2. **+Semantic chunking**: Same retrieval, but smarter chunking
3. **+Hybrid retrieval**: Fixed chunking, but combine dense + BM25
4. **+Full pipeline**: Semantic chunking + hybrid retrieval + reranking

In [ ]:
#@title 🎧 Listen: Ablation Code
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/05_28_ablation_code.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

In [ ]:
# Define the four configurations
configs = {
    'Baseline\n(Fixed + Dense)': {
        'chunk_size': 200,
        'chunk_overlap': 30,
        'chunking_method': 'fixed',
        'retrieval_method': 'dense',
        'use_reranking': False,
        'top_k': 3,
    },
    '+Semantic\nChunking': {
        'chunk_size': 200,
        'chunk_overlap': 30,
        'chunking_method': 'semantic',
        'retrieval_method': 'dense',
        'use_reranking': False,
        'top_k': 3,
    },
    '+Hybrid\nRetrieval': {
        'chunk_size': 200,
        'chunk_overlap': 30,
        'chunking_method': 'fixed',
        'retrieval_method': 'hybrid',
        'use_reranking': False,
        'top_k': 3,
    },
    'Full Pipeline\n(All Improvements)': {
        'chunk_size': 200,
        'chunk_overlap': 30,
        'chunking_method': 'semantic',
        'retrieval_method': 'hybrid',
        'use_reranking': True,
        'top_k': 3,
    },
}

# Use a subset of questions for the ablation (to save API calls)
ablation_questions = TEST_SET[:5]

print(f"Running ablation study with {len(ablation_questions)} questions "
      f"across {len(configs)} configurations...")
print(f"This will take several minutes due to LLM evaluation calls.\n")

ablation_results = {}

for config_name, config in configs.items():
    print(f"\n{'='*60}")
    print(f"Configuration: {config_name.replace(chr(10), ' ')}")
    print(f"{'='*60}")

    # Build pipeline
    pipe = RAGPipeline(KNOWLEDGE_BASE, config=config)
    pipe_stats = pipe.get_stats()
    print(f"  Chunks: {pipe_stats['num_chunks']}, "
          f"Avg length: {pipe_stats['avg_chunk_length']:.0f} words")

    config_scores = {'faithfulness': [], 'relevance': [], 'correctness': []}

    for i, test_case in enumerate(ablation_questions):
        answer, retrieved = pipe(test_case['question'])
        retrieved_texts = [r['text'] for r in retrieved]

        eval_result = evaluator.evaluate(
            question=test_case['question'],
            answer=answer,
            ground_truth=test_case['ground_truth'],
            retrieved_chunks=retrieved_texts
        )

        scores = eval_result['summary']
        config_scores['faithfulness'].append(scores['faithfulness'])
        config_scores['relevance'].append(scores['context_relevance'])
        config_scores['correctness'].append(scores['answer_correctness'])

        print(f"  Q{i+1}: F={scores['faithfulness']:.2f} "
              f"R={scores['context_relevance']:.2f} "
              f"C={scores['answer_correctness']:.2f}")

    ablation_results[config_name] = {
        'faithfulness': np.mean(config_scores['faithfulness']),
        'relevance': np.mean(config_scores['relevance']),
        'correctness': np.mean(config_scores['correctness']),
        'per_question': config_scores
    }

    print(f"\n  AVG: F={ablation_results[config_name]['faithfulness']:.3f} "
          f"R={ablation_results[config_name]['relevance']:.3f} "
          f"C={ablation_results[config_name]['correctness']:.3f}")

print("\n\nAblation study complete!")

In [ ]:
#@title 🎧 Listen: Ablation Viz
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/05_29_ablation_viz.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

In [ ]:
# Visualization: Stacked bar chart showing all 3 metrics across 4 configurations

config_names = list(ablation_results.keys())
faith_scores = [ablation_results[c]['faithfulness'] for c in config_names]
rel_scores = [ablation_results[c]['relevance'] for c in config_names]
corr_scores = [ablation_results[c]['correctness'] for c in config_names]

x = np.arange(len(config_names))
width = 0.55

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 7), gridspec_kw={'width_ratios': [2, 1]})

# Left: Grouped bar chart
bar_width = 0.22
bars1 = ax1.bar(x - bar_width, faith_scores, bar_width, label='Faithfulness',
                color='#1565C0', alpha=0.85, edgecolor='white')
bars2 = ax1.bar(x, rel_scores, bar_width, label='Context Relevance',
                color='#2E7D32', alpha=0.85, edgecolor='white')
bars3 = ax1.bar(x + bar_width, corr_scores, bar_width, label='Answer Correctness',
                color='#E65100', alpha=0.85, edgecolor='white')

ax1.set_ylabel('Score (0-1)', fontsize=12)
ax1.set_title('Pipeline Ablation: Impact of Each Improvement',
              fontsize=14, fontweight='bold')
ax1.set_xticks(x)
ax1.set_xticklabels(config_names, fontsize=9)
ax1.set_ylim(0, 1.15)
ax1.legend(fontsize=10, loc='upper left')
ax1.axhline(y=0.8, color='gray', linestyle='--', alpha=0.5)
ax1.text(len(config_names) - 0.5, 0.82, 'Target', fontsize=9, color='gray', style='italic')

# Value labels
for bars in [bars1, bars2, bars3]:
    for bar in bars:
        height = bar.get_height()
        ax1.text(bar.get_x() + bar.get_width()/2., height + 0.02,
                 f'{height:.2f}', ha='center', va='bottom', fontsize=8)

# Right: Overall score comparison
overall_scores = [(faith_scores[i] + rel_scores[i] + corr_scores[i]) / 3
                  for i in range(len(config_names))]
colors_overall = ['#90A4AE', '#42A5F5', '#66BB6A', '#FFA726']
short_names = ['Baseline', '+Semantic', '+Hybrid', 'Full']

bars_overall = ax2.barh(range(len(config_names)), overall_scores,
                         color=colors_overall, edgecolor='white', height=0.6)
ax2.set_xlim(0, 1.0)
ax2.set_yticks(range(len(config_names)))
ax2.set_yticklabels(short_names, fontsize=10)
ax2.set_xlabel('Overall Score (average of 3 metrics)', fontsize=11)
ax2.set_title('Overall Quality', fontsize=13, fontweight='bold')

for i, (bar, score) in enumerate(zip(bars_overall, overall_scores)):
    ax2.text(score + 0.02, i, f'{score:.3f}', ha='left', va='center', fontsize=10,
             fontweight='bold')

plt.tight_layout()
plt.show()

# Print improvement over baseline
print("\nImprovement over baseline:")
baseline_overall = overall_scores[0]
for i, name in enumerate(short_names[1:], 1):
    delta = overall_scores[i] - baseline_overall
    pct = (delta / baseline_overall) * 100 if baseline_overall > 0 else 0
    print(f"  {name}: {'+' if delta >= 0 else ''}{delta:.3f} "
          f"({'+' if pct >= 0 else ''}{pct:.1f}%)")

In [ ]:
#@title 🎧 Listen: Decision Framework Intro
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/05_30_decision_framework_intro.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

## 7. The Decision Framework

When should you use RAG versus fine-tuning versus long context? Let us build an interactive decision tree that takes your requirements and recommends the right approach.

In [ ]:
#@title 🎧 Listen: Decision Framework Code
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/05_31_decision_framework_code.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

In [ ]:
def decision_framework(knowledge_freshness: str = "dynamic",
                       corpus_size: str = "large",
                       latency_budget_ms: int = 2000,
                       need_citations: bool = True,
                       fine_tuning_budget: bool = False,
                       specialized_behavior: bool = False) -> Dict:
    """
    Recommend RAG vs Fine-Tuning vs Long Context based on requirements.

    Args:
        knowledge_freshness: "static" or "dynamic" (does knowledge change?)
        corpus_size: "small" (<100K tokens), "medium" (100K-1M), "large" (>1M)
        latency_budget_ms: Maximum acceptable latency in milliseconds
        need_citations: Whether source attribution is required
        fine_tuning_budget: Whether you have compute budget for fine-tuning
        specialized_behavior: Whether you need specialized tone/format
    """
    scores = {'RAG': 0, 'Fine-Tuning': 0, 'Long Context': 0}
    reasons = {'RAG': [], 'Fine-Tuning': [], 'Long Context': []}

    # Knowledge freshness
    if knowledge_freshness == "dynamic":
        scores['RAG'] += 3
        reasons['RAG'].append("Knowledge changes frequently — RAG can update without retraining")
        scores['Fine-Tuning'] -= 1
        reasons['Fine-Tuning'].append("Dynamic knowledge requires frequent retraining (expensive)")
    else:
        scores['Fine-Tuning'] += 1
        reasons['Fine-Tuning'].append("Stable knowledge can be baked into model weights")

    # Corpus size
    if corpus_size == "large":
        scores['RAG'] += 3
        reasons['RAG'].append("Large corpus exceeds any context window — retrieval is the only option")
        scores['Long Context'] -= 2
        reasons['Long Context'].append("Corpus too large for context window")
    elif corpus_size == "small":
        scores['Long Context'] += 3
        reasons['Long Context'].append("Small corpus fits in context window — simplest approach")
        scores['RAG'] += 1
        reasons['RAG'].append("Still works, but may be overengineered for small corpus")
    else:
        scores['RAG'] += 2
        reasons['RAG'].append("Medium corpus benefits from targeted retrieval")
        scores['Long Context'] += 1
        reasons['Long Context'].append("Might fit in modern context windows (128K-1M tokens)")

    # Latency
    if latency_budget_ms < 500:
        scores['Fine-Tuning'] += 2
        reasons['Fine-Tuning'].append("Tight latency budget — fine-tuned model has no retrieval overhead")
        scores['RAG'] -= 1
        reasons['RAG'].append("Retrieval adds latency (embedding + search + generation)")
    elif latency_budget_ms < 1000:
        scores['RAG'] += 1
        reasons['RAG'].append("Moderate latency budget is manageable for RAG")
    else:
        scores['RAG'] += 1
        reasons['RAG'].append("Generous latency budget — RAG retrieval is not a concern")

    # Citations
    if need_citations:
        scores['RAG'] += 3
        reasons['RAG'].append("RAG provides natural source attribution — critical for trust")
        scores['Fine-Tuning'] -= 1
        reasons['Fine-Tuning'].append("Fine-tuned models cannot cite specific sources reliably")

    # Fine-tuning budget
    if fine_tuning_budget and specialized_behavior:
        scores['Fine-Tuning'] += 2
        reasons['Fine-Tuning'].append("Budget available and specialized behavior needed")
    elif not fine_tuning_budget:
        scores['Fine-Tuning'] -= 2
        reasons['Fine-Tuning'].append("No fine-tuning budget — cannot pursue this approach")
        scores['RAG'] += 1
        reasons['RAG'].append("RAG requires no model training")

    # Specialized behavior
    if specialized_behavior:
        scores['Fine-Tuning'] += 2
        reasons['Fine-Tuning'].append("Specialized tone/format is best achieved through fine-tuning")

    # Determine recommendation
    recommendation = max(scores, key=scores.get)
    total = sum(max(0, v) for v in scores.values())
    confidence = {k: max(0, v) / total * 100 if total > 0 else 0 for k, v in scores.items()}

    return {
        'recommendation': recommendation,
        'scores': scores,
        'confidence': confidence,
        'reasons': reasons,
        'inputs': {
            'knowledge_freshness': knowledge_freshness,
            'corpus_size': corpus_size,
            'latency_budget_ms': latency_budget_ms,
            'need_citations': need_citations,
            'fine_tuning_budget': fine_tuning_budget,
            'specialized_behavior': specialized_behavior,
        }
    }

# Test with several scenarios
scenarios = [
    {
        'name': 'Customer Support Bot (dynamic docs, needs citations)',
        'params': dict(knowledge_freshness="dynamic", corpus_size="large",
                      latency_budget_ms=2000, need_citations=True,
                      fine_tuning_budget=False, specialized_behavior=False)
    },
    {
        'name': 'Medical Coding Assistant (specialized behavior, stable knowledge)',
        'params': dict(knowledge_freshness="static", corpus_size="medium",
                      latency_budget_ms=300, need_citations=False,
                      fine_tuning_budget=True, specialized_behavior=True)
    },
    {
        'name': 'Contract Analyzer (small corpus, cross-document reasoning)',
        'params': dict(knowledge_freshness="static", corpus_size="small",
                      latency_budget_ms=5000, need_citations=True,
                      fine_tuning_budget=False, specialized_behavior=False)
    },
]

print("Decision Framework Results")
print("=" * 70)

for scenario in scenarios:
    result = decision_framework(**scenario['params'])
    print(f"\nScenario: {scenario['name']}")
    print(f"  Recommendation: {result['recommendation']}")
    print(f"  Confidence: RAG={result['confidence']['RAG']:.0f}%, "
          f"Fine-Tuning={result['confidence']['Fine-Tuning']:.0f}%, "
          f"Long Context={result['confidence']['Long Context']:.0f}%")
    print(f"  Key reasons for {result['recommendation']}:")
    for reason in result['reasons'][result['recommendation']][:3]:
        print(f"    - {reason}")

In [ ]:
#@title 🎧 Listen: Decision Tree Viz
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/05_32_decision_tree_viz.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

In [ ]:
# Visualization: Decision tree plot

fig, ax = plt.subplots(1, 1, figsize=(16, 10))
ax.set_xlim(0, 16)
ax.set_ylim(0, 12)
ax.axis('off')
ax.set_title("RAG vs Fine-Tuning vs Long Context: Decision Framework",
             fontsize=15, fontweight='bold', pad=20)

def draw_node(ax, x, y, text, color, edge_color, width=2.8, height=0.9):
    box = plt.Rectangle((x - width/2, y - height/2), width, height,
                          linewidth=2, edgecolor=edge_color, facecolor=color,
                          zorder=3)
    ax.add_patch(box)
    ax.text(x, y, text, ha='center', va='center', fontsize=9,
            fontweight='bold', color=edge_color, zorder=4)

def draw_arrow(ax, x1, y1, x2, y2, label="", color='#333'):
    ax.annotate('', xy=(x2, y2), xytext=(x1, y1),
                arrowprops=dict(arrowstyle='->', color=color, lw=1.5),
                zorder=2)
    if label:
        mid_x = (x1 + x2) / 2
        mid_y = (y1 + y2) / 2
        ax.text(mid_x, mid_y + 0.15, label, ha='center', va='bottom',
                fontsize=8, color=color, style='italic',
                bbox=dict(boxstyle='round,pad=0.2', facecolor='white',
                          edgecolor='none', alpha=0.8))

# Root node
draw_node(ax, 8, 11, "Does knowledge\nchange frequently?", '#E3F2FD', '#1565C0', 3.0, 1.0)

# Level 1
draw_arrow(ax, 8, 10.5, 4, 9.5, "Yes (dynamic)", '#1565C0')
draw_arrow(ax, 8, 10.5, 12, 9.5, "No (static)", '#666')

draw_node(ax, 4, 9, "Need source\ncitations?", '#E8F5E9', '#2E7D32', 2.8, 0.9)
draw_node(ax, 12, 9, "Corpus fits in\ncontext window?", '#FFF3E0', '#E65100', 3.0, 0.9)

# Level 2 — left branch
draw_arrow(ax, 4, 8.55, 2.5, 7.5, "Yes", '#2E7D32')
draw_arrow(ax, 4, 8.55, 5.5, 7.5, "No", '#666')

draw_node(ax, 2.5, 7, "RAG", '#C8E6C9', '#1B5E20', 2.2, 0.8)
draw_node(ax, 5.5, 7, "Corpus size\n> 1M tokens?", '#E3F2FD', '#1565C0', 2.8, 0.9)

# Level 3 — left-right branch
draw_arrow(ax, 5.5, 6.55, 4.5, 5.5, "Yes", '#1565C0')
draw_arrow(ax, 5.5, 6.55, 7, 5.5, "No", '#666')

draw_node(ax, 4.5, 5, "RAG", '#C8E6C9', '#1B5E20', 2.2, 0.8)
draw_node(ax, 7, 5, "RAG or\nLong Context", '#FFF8E1', '#F57F17', 2.4, 0.8)

# Level 2 — right branch
draw_arrow(ax, 12, 8.55, 10, 7.5, "Yes", '#E65100')
draw_arrow(ax, 12, 8.55, 14, 7.5, "No", '#666')

draw_node(ax, 10, 7, "Need cross-doc\nreasoning?", '#FFF3E0', '#E65100', 2.8, 0.9)
draw_node(ax, 14, 7, "Need specialized\nbehavior/tone?", '#F3E5F5', '#6A1B9A', 3.0, 0.9)

# Level 3 — right branches
draw_arrow(ax, 10, 6.55, 9, 5.5, "Yes", '#E65100')
draw_arrow(ax, 10, 6.55, 11, 5.5, "No", '#666')

draw_node(ax, 9, 5, "Long\nContext", '#FFE0B2', '#E65100', 2.2, 0.8)
draw_node(ax, 11, 5, "RAG", '#C8E6C9', '#1B5E20', 2.2, 0.8)

draw_arrow(ax, 14, 6.55, 13, 5.5, "Yes", '#6A1B9A')
draw_arrow(ax, 14, 6.55, 15, 5.5, "No", '#666')

draw_node(ax, 13, 5, "Fine-\nTuning", '#E1BEE7', '#6A1B9A', 2.2, 0.8)
draw_node(ax, 15, 5, "Latency\n< 500ms?", '#FFEBEE', '#C62828', 2.4, 0.9)

# Level 4
draw_arrow(ax, 15, 4.55, 14, 3.5, "Yes", '#C62828')
draw_arrow(ax, 15, 4.55, 15.5, 3.5, "No", '#666')

draw_node(ax, 14, 3, "Fine-\nTuning", '#E1BEE7', '#6A1B9A', 2.2, 0.8)
draw_node(ax, 15.5, 3, "RAG", '#C8E6C9', '#1B5E20', 2.0, 0.8)

# Legend
legend_items = [
    (0.5, 1.4, '#C8E6C9', '#1B5E20', 'RAG: Retrieval-Augmented Generation'),
    (0.5, 0.6, '#E1BEE7', '#6A1B9A', 'Fine-Tuning: Specialized model training'),
    (8.0, 1.4, '#FFE0B2', '#E65100', 'Long Context: Full corpus in window'),
    (8.0, 0.6, '#FFF8E1', '#F57F17', 'Hybrid: Consider combining approaches'),
]

for x, y, fc, ec, label in legend_items:
    box = plt.Rectangle((x, y - 0.2), 0.4, 0.4, facecolor=fc, edgecolor=ec, linewidth=1.5)
    ax.add_patch(box)
    ax.text(x + 0.6, y, label, ha='left', va='center', fontsize=9)

plt.tight_layout()
plt.show()

print("\nKey insight: RAG is the default choice for most production scenarios.")
print("Fine-tuning wins when specialized behavior is critical and knowledge is stable.")
print("Long context wins when the full corpus fits and cross-document reasoning matters.")

In [ ]:
#@title 🎧 Listen: Final Output Intro
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/05_33_final_output_intro.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

## 8. Final Output

Let us run the full optimized pipeline on all 10 test questions, generate a comprehensive evaluation report, and identify every failure mode.

In [ ]:
#@title 🎧 Listen: Run Optimized Pipeline
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/05_34_run_optimized_pipeline.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

In [ ]:
# Build the best pipeline configuration (full pipeline)
best_config = {
    'chunk_size': 200,
    'chunk_overlap': 30,
    'chunking_method': 'semantic',
    'retrieval_method': 'hybrid',
    'use_reranking': True,
    'top_k': 3,
}

best_pipeline = RAGPipeline(KNOWLEDGE_BASE, config=best_config)
print("Full optimized pipeline built.")
print(f"Configuration: semantic chunking + hybrid retrieval + reranking")
print(f"Chunks: {len(best_pipeline.chunks)}")

print("\nRunning full evaluation on all 10 test questions...")
print("=" * 70)

final_results = []

for i, test_case in enumerate(TEST_SET):
    print(f"\n[Q{i+1}] {test_case['question']}")

    answer, retrieved = best_pipeline(test_case['question'])
    retrieved_texts = [r['text'] for r in retrieved]

    eval_result = evaluator.evaluate(
        question=test_case['question'],
        answer=answer,
        ground_truth=test_case['ground_truth'],
        retrieved_chunks=retrieved_texts
    )

    scores = eval_result['summary']

    # Diagnose failure mode if any metric is below 0.7
    failure_mode = "None"
    if scores['faithfulness'] < 0.7 or scores['context_relevance'] < 0.7 or scores['answer_correctness'] < 0.7:
        if scores['context_relevance'] < 0.5:
            failure_mode = "Wrong Retrieval"
        elif scores['faithfulness'] < 0.5:
            failure_mode = "Unfaithful Generation"
        elif scores['answer_correctness'] < 0.5:
            failure_mode = "Incorrect Answer"
        else:
            failure_mode = "Partial Degradation"

    final_results.append({
        'question': test_case['question'],
        'difficulty': test_case['difficulty'],
        'topic': test_case['topic'],
        'answer': answer,
        'ground_truth': test_case['ground_truth'],
        'eval': eval_result,
        'scores': scores,
        'failure_mode': failure_mode,
    })

    print(f"  Answer: {answer[:120]}...")
    print(f"  Faith: {scores['faithfulness']:.2f} | "
          f"Rel: {scores['context_relevance']:.2f} | "
          f"Corr: {scores['answer_correctness']:.2f} | "
          f"Failure: {failure_mode}")

In [ ]:
#@title 🎧 Listen: Generate Report
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/05_35_generate_report.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

In [ ]:
# Generate the comprehensive evaluation report

print("\n" + "=" * 70)
print("  COMPREHENSIVE RAG EVALUATION REPORT")
print("  Full Pipeline: Semantic Chunking + Hybrid Retrieval + Reranking")
print("=" * 70)

# Per-question metrics table
print(f"\n{'Q#':<4} {'Difficulty':<10} {'Faith':>7} {'Relev':>7} {'Corr':>7} {'Failure Mode':<22}")
print("-" * 65)

for i, r in enumerate(final_results):
    s = r['scores']
    print(f"Q{i+1:<3} {r['difficulty']:<10} {s['faithfulness']:>7.2f} "
          f"{s['context_relevance']:>7.2f} {s['answer_correctness']:>7.2f} "
          f"{r['failure_mode']:<22}")

# Aggregate scores
all_faith = [r['scores']['faithfulness'] for r in final_results]
all_rel = [r['scores']['context_relevance'] for r in final_results]
all_corr = [r['scores']['answer_correctness'] for r in final_results]

print("-" * 65)
print(f"{'AVG':<14} {np.mean(all_faith):>7.3f} {np.mean(all_rel):>7.3f} "
      f"{np.mean(all_corr):>7.3f}")
print(f"{'STD':<14} {np.std(all_faith):>7.3f} {np.std(all_rel):>7.3f} "
      f"{np.std(all_corr):>7.3f}")
print(f"{'MIN':<14} {np.min(all_faith):>7.3f} {np.min(all_rel):>7.3f} "
      f"{np.min(all_corr):>7.3f}")
print(f"{'MAX':<14} {np.max(all_faith):>7.3f} {np.max(all_rel):>7.3f} "
      f"{np.max(all_corr):>7.3f}")

# Failure mode analysis
print(f"\n{'='*50}")
print("  FAILURE MODE ANALYSIS")
print(f"{'='*50}")
failure_counts = Counter(r['failure_mode'] for r in final_results)
for mode, count in failure_counts.most_common():
    pct = count / len(final_results) * 100
    print(f"  {mode:<25} {count}/{len(final_results)} ({pct:.0f}%)")

# Score by difficulty
print(f"\n{'='*50}")
print("  SCORES BY DIFFICULTY")
print(f"{'='*50}")
for diff in ['easy', 'medium', 'hard']:
    subset = [r for r in final_results if r['difficulty'] == diff]
    if subset:
        avg_f = np.mean([r['scores']['faithfulness'] for r in subset])
        avg_r = np.mean([r['scores']['context_relevance'] for r in subset])
        avg_c = np.mean([r['scores']['answer_correctness'] for r in subset])
        print(f"  {diff:<8} (n={len(subset)}): Faith={avg_f:.3f}  "
              f"Rel={avg_r:.3f}  Corr={avg_c:.3f}")

# Recommendations
overall_avg = (np.mean(all_faith) + np.mean(all_rel) + np.mean(all_corr)) / 3
print(f"\n{'='*50}")
print("  RECOMMENDATIONS")
print(f"{'='*50}")
print(f"  Overall pipeline quality: {overall_avg:.3f}")

if np.mean(all_faith) < 0.7:
    print("  - FAITHFULNESS is below target. Consider:")
    print("    * Strengthening the system prompt to discourage hallucination")
    print("    * Adding explicit 'only use provided sources' instructions")
    print("    * Post-processing: claim verification before returning answers")

if np.mean(all_rel) < 0.7:
    print("  - CONTEXT RELEVANCE is below target. Consider:")
    print("    * Tuning chunk size and overlap parameters")
    print("    * Adding hybrid retrieval if not already using it")
    print("    * Implementing a reranker for better precision")

if np.mean(all_corr) < 0.7:
    print("  - ANSWER CORRECTNESS is below target. Consider:")
    print("    * Checking if retrieved context contains the right information")
    print("    * Reviewing knowledge base coverage for question topics")
    print("    * Improving generation prompt to better extract relevant details")

if overall_avg >= 0.7:
    print("  - Pipeline meets minimum quality bar (>= 0.70)")
    print("  - Focus on improving the weakest metric for further gains")

print(f"\n{'='*70}")
print("  END OF REPORT")
print(f"{'='*70}")

In [ ]:
#@title 🎧 Listen: Final Summary Viz
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/05_36_final_summary_viz.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

In [ ]:
# Final summary visualization

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# 1. Radar-style metric comparison (using bar chart)
metrics = ['Faithfulness', 'Context\nRelevance', 'Answer\nCorrectness']
avg_scores = [np.mean(all_faith), np.mean(all_rel), np.mean(all_corr)]
colors_metric = ['#1565C0', '#2E7D32', '#E65100']

axes[0].bar(metrics, avg_scores, color=colors_metric, alpha=0.85,
            edgecolor='white', linewidth=2)
axes[0].set_ylim(0, 1.0)
axes[0].set_title('Aggregate Metric Scores', fontsize=12, fontweight='bold')
axes[0].axhline(y=0.8, color='red', linestyle='--', alpha=0.5)
axes[0].text(2.5, 0.82, 'Target', fontsize=8, color='red', style='italic')
for i, (score, color) in enumerate(zip(avg_scores, colors_metric)):
    axes[0].text(i, score + 0.02, f'{score:.3f}', ha='center', va='bottom',
                 fontsize=11, fontweight='bold', color=color)

# 2. Score distribution (box plots)
data_for_box = [all_faith, all_rel, all_corr]
bp = axes[1].boxplot(data_for_box, labels=['Faith', 'Relevance', 'Correct'],
                      patch_artist=True, widths=0.5)
for patch, color in zip(bp['boxes'], colors_metric):
    patch.set_facecolor(color)
    patch.set_alpha(0.4)
axes[1].set_ylim(0, 1.1)
axes[1].set_title('Score Distribution', fontsize=12, fontweight='bold')
axes[1].set_ylabel('Score')

# 3. Failure mode pie chart
if len(failure_counts) > 1 or list(failure_counts.keys())[0] != "None":
    labels_pie = list(failure_counts.keys())
    sizes_pie = list(failure_counts.values())
    colors_pie = ['#81C784' if l == 'None' else '#EF5350' if 'Wrong' in l
                  else '#FFA726' if 'Unfaith' in l else '#42A5F5'
                  for l in labels_pie]
    axes[2].pie(sizes_pie, labels=labels_pie, colors=colors_pie, autopct='%1.0f%%',
                textprops={'fontsize': 9}, startangle=90)
    axes[2].set_title('Failure Mode Distribution', fontsize=12, fontweight='bold')
else:
    axes[2].text(0.5, 0.5, 'No failures\ndetected!', ha='center', va='center',
                 fontsize=16, fontweight='bold', color='#2E7D32',
                 transform=axes[2].transAxes)
    axes[2].set_title('Failure Mode Distribution', fontsize=12, fontweight='bold')

plt.tight_layout()
plt.show()

print("\nCapstone evaluation complete.")
print("You now have a fully evaluated, end-to-end RAG system with diagnostic tools.")

In [ ]:
#@title 🎧 Listen: Reflection
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/05_37_reflection.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

## 9. Reflection and Next Steps

### Reflection Questions

Take a moment to think about what you have built and what you have learned.

**Question 1:** We used LLM-as-judge for faithfulness and context relevance scoring. This means we are using an LLM to evaluate the output of another LLM. What are the failure modes of this approach? When might the judge itself hallucinate or make mistakes? How would you validate that your evaluator is reliable?

**Question 2:** The pipeline ablation study tested four configurations. But we only varied three dimensions (chunking, retrieval, reranking). What other dimensions could you vary? Think about: chunk size, overlap amount, number of retrieved chunks (top-k), embedding model choice, generation prompt structure, temperature. How would you design an ablation study that tests all of these efficiently?

**Question 3:** Our decision framework is rule-based. Could you build a more sophisticated version that uses actual benchmark results to recommend approaches? What data would you need to collect, and how would you handle the fact that the "best" approach depends on the specific domain and use case?

### Optional Challenges

**Challenge 1 — Adversarial Evaluation:** Create 5 adversarial test questions designed to break the pipeline. These might include: questions that require combining information from multiple chunks, questions with ambiguous phrasing, questions where the answer is not in the knowledge base at all, and questions where similar-sounding but different concepts could cause confusion. Run the evaluation and analyze the failure modes.

**Challenge 2 — Confidence Calibration:** Modify the pipeline to output a confidence score with each answer. The confidence should be based on retrieval scores, the consistency of retrieved chunks, and the model's own uncertainty signals. Then measure whether the confidence score is well-calibrated — are answers with 90% confidence actually correct 90% of the time?

**Challenge 3 — Evaluation Without Ground Truth:** In production, you often do not have ground-truth answers. Design an evaluation framework that works without them, using only faithfulness and context relevance (which do not require ground truth). How well do these two metrics alone predict answer quality? Run the experiment and report the correlation.

### Series Summary

Over five notebooks, you have built a complete RAG system from first principles:

| Notebook | What You Built | Key Insight |
|----------|---------------|-------------|
| **1. Foundations** | Why retrieval matters, the RAG equation | $P(y \mid x, z)$ — conditioning on retrieved documents transforms accuracy |
| **2. Chunking and Embeddings** | Document processing, semantic representations | Chunk boundaries determine retrieval quality; embeddings capture meaning |
| **3. Indexing and Retrieval** | Vector databases, dense/sparse/hybrid search | Two-stage retrieval (fast recall + precise reranking) is the production pattern |
| **4. Generation and Prompting** | Context assembly, faithfulness instructions | The prompt determines whether the model uses or ignores the retrieved context |
| **5. Evaluation (this notebook)** | RAGAS metrics, ablation studies, decision framework | A system that cannot be measured cannot be improved |

The core lesson across all five notebooks is this: RAG is not "search plus LLM." It is a carefully designed pipeline where every component — from chunk boundaries to embedding models to retrieval strategies to prompt structure to evaluation metrics — compounds to determine the quality of the final answer. Get any one step wrong, and the whole system suffers. Get them all right, and you have a system that is accurate, verifiable, and perpetually up to date.

You now have the tools to build, evaluate, and diagnose RAG systems. Go build something real.